In [ ]:
import csv
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import sys
import scipy.signal as signal
from scipy.stats import norm  # for u(t) as gaussians

from scipy.ndimage import gaussian_filter1d
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

Matplotlib settings

In [ ]:
# help with Illustrator fonts
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['ps.fonttype'] = 42

color_input = '#00AEEF'
#color_memory = '#DEB87A'  
color_memory = '#C1A16B'  # dark: #DEB87A || darker: #D0AD73 ||| darkest: #C1A16B
color_response = '#662D91'

color_1 = '#663290'           # regular y(t) purp OR 'darkblue'
color_2 = '#8965AC'           # medium purp       OR 'royalblue'
color_3 = '#B19CC9'           # light purp        OR 'cornflowerblue'

linewidth = 1.0
linestyle = '-'

line_kwargs_input = dict(color=color_input, linewidth=linewidth, linestyle=linestyle)
line_kwargs_memory = dict(color=color_memory, linewidth=linewidth, linestyle=linestyle)
#line_kwargs_memory_sigma = dict(color=color_memory, linewidth=linewidth, linestyle='dotted')
line_kwargs_memory_sigma = dict(color=color_memory, linewidth=linewidth, linestyle=linestyle)
line_kwargs_response = dict(color=color_response, linewidth=linewidth, linestyle=linestyle)

ls_kwargs=dict(
    markersize=3,
    marker='o',
    linestyle='-',
    linewidth=1,
)

### Directory paths

In [ ]:
SRC_ROOT = os.path.dirname(os.path.abspath(''))
print('appending to path SRC_ROOT...\n\t', SRC_ROOT)
sys.path.append(SRC_ROOT)

PACKAGE_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('')))
print('appending to path PACKAGE_ROOT...\n\t', PACKAGE_ROOT)
sys.path.append(PACKAGE_ROOT)

# particular io folders
NB_OUTPUT = SRC_ROOT + os.sep + 'output'
if not os.path.exists(NB_OUTPUT):
    os.makedirs(NB_OUTPUT)

## 1. (Experimental data) Functions to load data

In [ ]:
from python.data_tools import DIR_INPUT, build_mergetrials_dataframe, build_dataframe_from_data, filter_df_by_filterdict, df_to_arr_jumps

from python.settings import heatmap_0, heatmap_1

## 2. (Experimental data) Load all datafiles into single pd DataFrame 

In [ ]:
data_version = 'v3'  # select v2 or v3 (April 2024)

df_full = build_dataframe_from_data(data_version=data_version)

In [ ]:
# settings depend on experimental setup version (v2 or v3)

if data_version == 'v2':
    n_pulse_hab = 100
    n_pulse_reactivity = 50
    # settings for synthetic dataset generation
    num_flies = 128  # could be bigger then just truncate... to 128
    num_hab_cycles = 3
    num_pulse_hab = 100
    period_S1_hab        = 1.0  # period between consecutive pulses
    period_S1_reactivity = 5.0  # period between consecutive pulses
    num_pulse_reactivity = 50
else:
    assert data_version == 'v3'
    n_pulse_hab = 200        # defines v3 data
    n_pulse_reactivity = 50  # defines v3 data
    # settings for synthetic dataset generation
    num_flies = 128
    num_hab_cycles = 5
    num_pulse_hab = 200
    period_S1_hab        = 1.0  # period between consecutive pulses
    period_S1_reactivity = 5.0  # period between consecutive pulses
    num_pulse_reactivity = 50

In [ ]:
df_full

In [ ]:
df_full.dtypes

In [ ]:
df_full_mergetrials = build_mergetrials_dataframe(df_full, data_version=data_version)

## 3. (Experimental data) Inspect basic stats from the full dataframe

In [ ]:
df_full.head()

## 4. (Experimental data) Plot of jump events and mean jumps at time t for partial dataset (GD or KK background)

In [ ]:
#df_to_use = df_full_mergetrials
#df_to_use = filter_df_by_filterdict(df_full_mergetrials, dict(gene_bgr=['GD']))
df_to_use = filter_df_by_filterdict(df_full_mergetrials, dict(gene_bgr=['KK']))

In [ ]:
jumps = df_to_arr_jumps(df_to_use)
M, n = jumps.shape
cmap_custom = mpl.colors.ListedColormap([heatmap_0, heatmap_1])  # no data (-1) | no jump (0) | jump (1)
aspect = n/M*0.6

plt.figure(figsize=(6,12))
plt.imshow(jumps, aspect=aspect, interpolation='None', cmap='viridis')
#plt.imshow(jumps, aspect=aspect, interpolation='None', cmap=cmap_custom)
plt.show()

print('Data shape')
print('\tfull:', jumps.shape)

In [ ]:
jumps_sums = np.sum(jumps, axis=0) / M
print('\tcolumn sums:', jumps_sums.shape)

plt.figure(figsize=(7,2))
plt.plot(np.arange(n), jumps_sums, '--ok', linewidth=1, markersize=2)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.title('mean jumps at each timestep')
plt.ylim(0,0.65)

if data_version == 'v2':

    jump_sums_1a = jumps_sums[0:n_pulse_hab]
    jump_sums_1b = jumps_sums[n_pulse_hab : 2 * n_pulse_hab]
    jump_sums_1c = jumps_sums[2 * n_pulse_hab : 3 * n_pulse_hab]
    jump_sums_5 = jumps_sums[3 * n_pulse_hab:]
    
    plt.figure(figsize=(8.5,3))
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1a, '-o', c='#4D4DFF', linewidth=1, markersize=2, label=r'$T=1s$ (expt 1/3)')
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1b, '-o', c='#7B68EE', linewidth=1, markersize=2, label=r'$T=1s$ (expt 2/3)')
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1c, '-o', c='#76ABDF', linewidth=1, markersize=2, label=r'$T=1s$ (expt 3/3)')
else:
    jump_sums_1a = jumps_sums[0:n_pulse_hab]
    jump_sums_1b = jumps_sums[n_pulse_hab : 2 * n_pulse_hab]
    jump_sums_1c = jumps_sums[2 * n_pulse_hab : 3 * n_pulse_hab]
    jump_sums_1d = jumps_sums[3 * n_pulse_hab : 4 * n_pulse_hab]
    jump_sums_1e = jumps_sums[4 * n_pulse_hab : 5 * n_pulse_hab]
    jump_sums_5 = jumps_sums[5 * n_pulse_hab:]
    
    plt.figure(figsize=(8.5,3))
    alpha_hab_expt = 1.0
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1a, '-o', c='#4D4DFF', linewidth=1, markersize=2, label=r'$T=1s$ (Hab 1)', alpha=alpha_hab_expt)
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1b, '-o', c='#7B68EE', linewidth=1, markersize=2, label=r'$T=1s$ (Hab 2)', alpha=alpha_hab_expt)
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1c, '-o', c='#76ABDF', linewidth=1, markersize=2, label=r'$T=1s$ (Hab 3)', alpha=alpha_hab_expt)
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1d, '-o', c='#76ABDF', linewidth=1, markersize=2, label=r'$T=1s$ (Hab 4)', alpha=alpha_hab_expt)
    plt.plot(np.arange(0, n_pulse_hab), jump_sums_1e, '-o', c='#76ABDF', linewidth=1, markersize=2, label=r'$T=1s$ (Hab 5)', alpha=alpha_hab_expt)
    
plt.plot(np.arange(0, n_pulse_reactivity), jump_sums_5, '-ok', linewidth=1, markersize=2, label=r'$T=5s$')
plt.xlabel(r'Stimulus index $k$')
#plt.ylabel(r'Fraction jump at $k$')
plt.title(r'Fraction of $M=%d$ flies that jump at $k^{th}$ shadow' % M)
plt.legend()

# Functions for generating filter ODE (to create synthetic jump data)

In [ ]:
def signal_rectangle(n_pulse, n_rest, n_cycles, times=None, period_S1=1.0, duty=0.01, pulse_area=1.0):
    
    def ubase_rect(t, t_mid, duty, period_S1):
        scale_for_unit_area = 1 / (duty * period_S1)
        val_at_t = scale_for_unit_area * (
            np.heaviside(t - (t_mid - duty * period_S1), 0) - 
            np.heaviside(t - t_mid, 0))
        return val_at_t
    
    L_cycle = n_pulse + n_rest
    
    if times is None:
        dt = 0.001
        tmax = L_cycle * n_cycles * period_S1
        times = np.arange(-2, tmax + dt, dt)
    else:
        dt = times[1] - times[0]
        
    tmid_list = [period_S1 * (n + L_cycle * k) for k in range(n_cycles) for n in range(1, n_pulse+1)]
    
    # build u: wave of rectangles
    assert duty * period_S1 > 5 * dt  # want it to be sampled nicely
    u_rect = pulse_area * np.sum([ubase_rect(times, t_mid, duty, period_S1) for t_mid in tmid_list], axis=0)   
    
    return times, u_rect


def signal_gaussian(n_pulse, n_rest, n_cycles, period_S1=1.0, times=None, sigma=0.05, pulse_area=1.0):
    
    def ubase_gaussian(t, t_mid, sigma):
        # direct calc has exp overflow
        '''sigma_sqr = sigma ** 2 
        prefactor = 1 / np.sqrt(2 * np.pi * sigma_sqr)
        expval = (t - t_mid) ** 2 / (2 * sigma_sqr)
        val_at_t = prefactor * np.exp(expval)'''
        val_at_t = norm.pdf(t, t_mid, sigma)
        return val_at_t
    
    L_cycle = n_pulse + n_rest
    
    if times is None:
        dt = 0.001
        tmax = L_cycle * n_cycles * period_S1
        times = np.arange(-2, tmax + dt, dt)
    else:
        dt = times[1] - times[0]
    
    tmid_list = [period_S1*(n + L_cycle * k) for k in range(n_cycles) for n in range(1, n_pulse+1)]
    
    # build u: wave of rectangles
    u_gaussian = pulse_area * np.sum([ubase_gaussian(times, t_mid, sigma) for t_mid in tmid_list], axis=0)
    
    return times, u_gaussian

In [ ]:
def W_of_t_convolve(t, u, rate_decay, rate_grow, nmult=5):
    "As opposed to direct integral in W_of_t_manual, use convolution"    
    dt = t[1] - t[0]
    nn = len(t)
    
    # size of window depends on alpha, slower (smaller) alpha means longer window
    tsample_exp_window = np.arange(0, 
                                   (nmult * nn) * dt + dt, 
                                   dt)
    exp_of_t_window = np.exp(-rate_decay * tsample_exp_window)
    print('nn=len(t) == %d, while len(tsample_exp_window) == %d; (20/rate_decay/dt) =' % (nn, len(tsample_exp_window) ), (20/rate_decay/dt))
    assert len(tsample_exp_window) > nn
    
    #W_of_t = dt * alpha * np.convolve(u, exp_of_t_window, 'same')  
    #W_of_t = W_of_t[0:nn]  # truncate to early values matching length of u
    
    W_of_t = dt * rate_grow * signal.convolve(u, exp_of_t_window, 'full', method='fft')  
    W_of_t = W_of_t[0:nn]  # truncate to early values matching length of u
    
    return W_of_t

In [ ]:
def plot_uy_stack(t, u, y, title, fpath, u_shadow=True, simplify=False, figsize=(3, 2), xlims=None):
    """
    Args:
        fmod: 'rectangle', 'gaussian'
    """
    # plot params
    grid_alpha = 0.6

    plt.close(); 
    ###nrows = 2
    ###fig, axarr = plt.subplots(nrows, 1, sharex=True, squeeze=False, figsize=figsize)
    fig = plt.figure(figsize=figsize)
    fig.suptitle(title)
    
    gs = GridSpec(2, 1, height_ratios=[1, 2], hspace=0.2)
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1], sharex=ax0)
    axarr = [ax0, ax1]
    
    #axarr[0,0].set_title(r'$foo 1$')
    axarr[0].plot(t, u, label=r'$u(t)$', **line_kwargs_input)

    #axarr[-1,0].set_title(r'foo 4')
    axarr[1].plot(t, y, label=r'$y(t)=u(t) r(t)$', **line_kwargs_response)
    
    for idx, ax in enumerate(axarr):
        if idx > 0 and u_shadow:
            ax.plot(t, u, '-k', alpha=0.1)
        if xlims is not None:
            ax.set_xlim(xlims)
    
    if simplify:
        ylabels = ['input', 'response']
        fprop = {'family': 'arial', 'size': 12}
        for idx, ax in enumerate(axarr):
            ax.set_ylabel(ylabels[idx], **fprop)
            ax.tick_params(labelbottom=False, labelleft=False)
            ax.spines[['right', 'top']].set_visible(False)
            ax.tick_params(which='both', size=0, labelsize=0)
        axarr[-1].set_xlabel('t', loc='right', style='italic', family='arial', size=12)
    else:
        fprop = {'family': 'arial', 'size': 12}
        ax0.set_ylabel(r'input $u(t)$')
        ax1.set_ylabel(r'output $y(t)$')
        for idx, ax in enumerate(axarr):
            #ax.grid(alpha=grid_alpha)
            ax.legend()
        axarr[-1].set_xlabel(r'$t$', loc='right', **fprop)
    
    print(fpath)
    plt.savefig(fpath + '.pdf')
    plt.savefig(fpath + '.svg')
    plt.show()
    
    '''
    def annotate_axes(fig):
    for i, ax in enumerate(fig.axes):
        ax.text(0.5, 0.5, "ax%d" % (i+1), va="center", ha="center")
        ax.tick_params(labelbottom=False, labelleft=False)
    '''
    return axarr

def plot_uxrp_stack(t, u, x, r, p, title, fpath):
    """
    Args:
        fmod: 'rectangle', 'gaussian'
    """
    # plot params
    grid_alpha = 0.6

    plt.close(); 
    fig = plt.figure(figsize=(7, 7))
    #fig, axarr = plt.subplots(4, 1, sharex=True, squeeze=False, figsize=(7, 7))
    
    gs = GridSpec(4, 1, height_ratios=[1, 2, 2, 2], hspace=0.25)
    
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1], sharex=ax0)
    ax2 = fig.add_subplot(gs[2], sharex=ax0)
    ax3 = fig.add_subplot(gs[3], sharex=ax0)
    axarr = [ax0, ax1, ax2, ax3]
    
    axarr[0].set_title(title)
    axarr[0].set_ylabel(r'input $u(t)$')
    axarr[1].set_ylabel(r'memory $x(t)$')
    axarr[2].set_ylabel(r'receptivity $r(t)$')
    axarr[3].set_ylabel(r'jump $p(t)$')
    axarr[-1].set_xlabel(r'$t$')
    
    axarr[0].plot(t, u, label=r'$u(t)$')
    axarr[0].set_ylim(-0.05, np.max(u)*1.05)

    #axarr[1,0].set_title(r'$foo 2$')
    axarr[1].plot(t, x, label=r"$x(t)=\int_{0}^t \,\beta e^{-\alpha (t-t')} u(t') dt'$")
    axarr[1].axhline(1, linestyle='--', alpha=0.5)
    axarr[1].set_ylim(-0.05, np.max(x_of_t)*1.05)

    #axarr[2,0].set_title(r'$foo 3$')
    axarr[2].plot(t, r, label=r'$r(t)=\sigma(x)$')
    axarr[2].axhline(0.5 * np.max(u), linestyle='--', alpha=0.5)

    #axarr[3,0].set_title(r'foo 4')
    axarr[3].plot(t, p, label=r'$p_{jump}(t) = p_0 u(t) r(t)$')
    axarr[3].axhline(0.5, linestyle='--', alpha=0.5)

    for idx in range(0, 4):
        #if idx > 0:
        #    axarr[idx,0].plot(t, u, '-k', alpha=0.1)
        #axarr[idx,0].grid(alpha=grid_alpha)
        axarr[idx].legend()

    plt.savefig(fpath)
    plt.show()
    return axarr

In [ ]:
# From the timeseries y_of_t, sample a probability at integer times dt=1
# - currently unused, we sample the ODE output at regular intervals k*T
def convert_cts_to_discrete_avg(t, y_cts, t_intervals):
    """
    Check if uniform t sampled version needed
    
    Main idea here: during each interval over which the stimulus is non-zero, take the mean of y_cts
    - an alternative: take max over interval
    """
    y_discretized = np.zeros(len(t_intervals) - 1)
    t_discretized = np.zeros(len(t_intervals) - 1)
    t_coords = np.searchsorted(t, t_intervals)
    print(t_coords)
    for idx in range(len(t_intervals)-1):
        t_idx_0 = t_coords[idx]    # i.e. pos of 0.0, 1.0, etc
        t_idx_1 = t_coords[idx+1]  # i.e. pos of 1.0, 2.0, etc
        t_discretized[idx] = np.mean(t[t_idx_0 : t_idx_1])
        y_discretized[idx] = np.mean(y_cts[t_idx_0 : t_idx_1])
    return y_discretized, t_discretized

# Now consider different models to generate synthetic data

In [ ]:
def gen_jumps_from_prob_flips(nflips, nrepeats, pflips):

    jump_arr = np.zeros((nrepeats, nflips))  # rows are samples, columns are individual timepoints

    for k in range(nrepeats):
        for idx in range(nflips):
            urand = np.random.rand()
            if urand < pflips[idx]:
                jump_arr[k, idx] = 1
            else:
                jump_arr[k, idx] = 0
    return jump_arr


### Example 1: coin flips with fixed probability

In [ ]:
nflips = num_pulse_hab
nrepeats = 40
pflips_fixed = 0.2 * np.ones(nflips)
jump_arr = gen_jumps_from_prob_flips(nflips, nrepeats, pflips_fixed)

plt.close()
plt.imshow(jump_arr, interpolation='none')
plt.xlabel('time')
plt.ylabel('repeats')
plt.title('fixed probability to jump (%.2f)' % pflips_fixed[0])
plt.show()

### Example 2: Have decreasing p_jump(t) using pulses into wiener model

#### 2.1 - model parameters

In [ ]:
# model parameters
model_memory_alpha = 0.1
model_memory_beta  = 0.2
model_jump_p0      = 0.6  # this is virgin jump probability

#### 2.2 - stimulus settings and assumptions

In [ ]:
period_S1 = 1.0 # period between consecutive pulses
n_rest = 120
L_cycle = num_pulse_hab + n_rest
n_cycles = 1  # 3
tmid_list = [period_S1*(n + L_cycle * k) for k in range(num_hab_cycles) for n in range(1, num_pulse_hab+1)]

# prep t to sample signals from
dt = 0.001
#tmax = L_cycle * period_S1 * 4
tmax = L_cycle * num_hab_cycles * period_S1
t = np.arange(-0.01, tmax + dt, dt)

# choose pulse_area
pulse_area = 1.0
duty = 0.01 / period_S1

# build u: wave of rectangles
_, u_rect = signal_rectangle(num_pulse_hab, n_rest, num_hab_cycles, period_S1=1.0, times=t, duty=duty, pulse_area=1.0)

u = u_rect

#### 2.3 - demonstrate model attentuation

In [ ]:
x_of_t = W_of_t_convolve(t, u, model_memory_alpha, model_memory_beta)
y_of_t = u / (1 + x_of_t ** 2)

"""fpath = NB_OUTPUT + os.sep + 'uy_stack_comb_identity'
_ = plot_uy_stack(t, u, y_of_t, '', fpath, simplify=False, figsize=(5, 2))"""

#### 2.4 - naive example of conversion of y(t) to a probability (rate) timeseries

In [ ]:
jumprate_timeseries = model_jump_p0 * y_of_t/100

plt.figure(figsize=(6, 1))
plt.plot(t, jumprate_timeseries, zorder=10)
#plt.plot(t, u, alpha=0.2, zorder=1)
plt.ylim(-0.01, 1.1)
plt.xlabel(r'$t$') 
plt.ylabel(r'$p(t)$')
plt.title(r'$\alpha=%.2f, \beta=%.2f, p_0=%.2f$' % (model_memory_alpha, model_memory_beta, model_jump_p0))

In [ ]:
t_at_pulses = np.arange(period_S1,
                        period_S1 * num_pulse_hab + period_S1,
                        period_S1) - 0.99 * duty * period_S1
indices = np.searchsorted(t, t_at_pulses)
t_sub = t[indices]
x_sub = x_of_t[indices]


plot_debugging = False
if plot_debugging:

    t_intervals = np.arange(0, num_pulse_hab * period_S1 + period_S1, period_S1)
    y_discrete, t_discrete = convert_cts_to_discrete_avg(t, y_of_t, t_intervals)
    
    """plt.plot(t_discrete, y_discrete, '--o', label='avg method')
    plt.title("plt.plot(t_discrete, y_discrete, '--o')")
    plt.legend()
    plt.show()"""
    
    # ========================================
    plt.plot(t, x_of_t)
    plt.plot(t_sub, x_sub, 'o', markersize=4, label=r'x_sub (val at specific time)')
    plt.plot(t_sub, 1 / (1 + x_sub ** 2), 'o', markersize=4, label=r'$\sigma(x)$')
    plt.plot(t_discrete+0.5, y_discrete, '--o', label='$\sigma(x)$ avg method')
    plt.plot(t_sub, 1 / (1 + np.exp(x_sub - 1)), 'o', markersize=4, label=r'$\sigma(x)$ alt')
    plt.title('plt.plot(t, x_of_t) and discrete variants')
    plt.xlim(-0.2, 75)
    plt.legend()
    plt.show()
    
    """plt.plot(t, u); plt.xlim(0,5)
    plt.title('plt.plot(t, u); plt.xlim(0,5)')
    plt.show()"""

In [ ]:
if plot_debugging:
    fpath = NB_OUTPUT + os.sep + 'uxrp_model'
    title = r'Wiener model dynamics ($\alpha=%.1f,\beta=%.1f, p_0=%.2f, T=%.2f$)' % (model_memory_alpha, model_memory_beta, model_jump_p0, period_S1)
    #_ = plot_uy_stack(t, u, y_of_t, '', fpath, simplify=False, figsize=(5,2))
    
    r_of_t = 1 / (1 + x_of_t ** 2)
    p_of_t = model_jump_p0 * r_of_t * u / 100
    plot_uxrp_stack(t, u/100, x_of_t, r_of_t, p_of_t, title, fpath)

#### Now use time-dependent prob to flip using output of dyn sys

In [ ]:
pbase = 0.6  # this comes from training data...

pflips_variable = pbase * 1/(1+x_sub**2)
jump_arr = gen_jumps_from_prob_flips(num_pulse_hab, nrepeats, pflips_variable)

In [ ]:
plt.close()
fig = plt.figure(figsize=(5,5))
gs_main = GridSpec(2, 1, height_ratios=[1, 1], hspace=0.45)

gs1 = GridSpecFromSubplotSpec(1, 1, subplot_spec=gs_main[0])
gs2 = GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[1], hspace=.15)

ax0 = fig.add_subplot(gs1[0])
ax1 = fig.add_subplot(gs2[0], sharex=ax0)
ax2 = fig.add_subplot(gs2[1], sharex=ax0)

ax0.imshow(jump_arr, interpolation='none', aspect='auto')
ax0.set_title(r'Synthetic data $(T=%.2f)$' % period_S1)


ax1.plot(t_sub/period_S1, pflips_variable, '-og', markersize=3, label=r'hidden $p_{jump}$')
ax1.legend()
ax1.set_ylim(-0.01,1.05)
ax1.set_xlim(-0.5, num_pulse_hab - 0.5)
ax1.set_title(r'habituation of probability to jump ($%.2f$ to $%.2f$)' % (pflips_variable[0], pflips_variable[-1]))
ax1.tick_params(labelbottom=False)

ax2.plot(t_sub/period_S1, np.mean(jump_arr, axis=0), '--ok', markersize=3, label=r'mean $\langle z_k \rangle$')
ax2.legend()
ax2.set_ylim(-0.01,1.05)
ax2.set_xlim(-0.5, num_pulse_hab - 0.5)

ax2.set_xlabel(r'time $t/T$')

#gs.tight_layout(fig, rect=[0, 0, 1, 0.97])
plt.show()

## Now use time-dependent prob to flip using output of dyn sys

#### Note: total number of jumps (for same age, gene bgr) in the dt=5 case may be compared vs binomial distributon (iid coins)

In [ ]:
nrepeats = 500  # this is number of flies for each parameter set

pflips_variable_example = model_jump_p0 * 1 / (1 + x_sub ** 2)
jump_arr = gen_jumps_from_prob_flips(num_pulse_reactivity, nrepeats, pflips_variable_example)

### 2.3 - Now create merged experiment with multiple 100x or 200x trials of T=1s and 50x of T=5s (with gap of 120s in between)

#### construct the stimulus

In [ ]:
duration_rest = 120  # 2 min
n_rest = 120
assert period_S1_hab == 1.0  # hardcoded duration rest 120
L_cycle = num_pulse_hab + n_rest

tmid_list = [period_S1_hab * (n + L_cycle * k) for k in range(num_hab_cycles) for n in range(1, num_pulse_hab+1)]

# prep t to sample signals from
dt = 0.001

#tmax = L_cycle * period_S1 * 4
tmax = L_cycle * num_hab_cycles * period_S1_hab
t = np.arange(-0.01, tmax + dt, dt)

tmax_reactivity = (num_pulse_reactivity + 1) * period_S1_reactivity
t_reactivity = np.arange(-0.01, tmax_reactivity + dt, dt)

# choose pulse_area
pulse_area = 1.0
duty_hab = 0.01 / period_S1_hab
duty_reactivity = 0.01 / period_S1_reactivity

# build u: wave of rectangles
_, u_rect_hab = signal_rectangle(num_pulse_hab, n_rest, num_hab_cycles, period_S1=period_S1_hab, times=t, duty=duty_hab, pulse_area=pulse_area)
_, u_rect_reactivity = signal_rectangle(num_pulse_reactivity, 1, 1, period_S1=period_S1_reactivity, times=t_reactivity, duty=0.01 / period_S1_reactivity, pulse_area=pulse_area)

In [ ]:
x_of_t = W_of_t_convolve(t_reactivity, u_rect_reactivity, model_memory_alpha, model_memory_beta)
y_of_t = u_rect_reactivity / (1 + x_of_t ** 2)

#fpath = NB_OUTPUT + os.sep + 'uy_stack_comb_identity'
#_ = plot_uy_stack(t_reactivity, u_rect_reactivity, y_of_t, '', fpath, simplify=False, figsize=(5,2))

In [ ]:
#t_merge = np.concatenate([t, t[-1] + t_reactivity])
mergebuffer = 1 + int(0.01/dt)
t_merge = np.concatenate([t, 
                          t[-1] + t_reactivity[mergebuffer:]])

u_merge = np.concatenate([u_rect_hab, 
                          u_rect_reactivity[mergebuffer:]])

"""
plt.plot(t_merge, u_merge)
plt.title('u_merge')
plt.show()"""

## Now use this standard merged input $u(t)$ to generate different synthetic datasets

In [ ]:
def ode_response_given_t_u_theta_v1(t_merge, u_merge, model_memory_alpha, model_memory_beta, model_jump_p0, nmult=5):
    x_of_t = W_of_t_convolve(t_merge, u_merge, model_memory_alpha, model_memory_beta, nmult=nmult)
    #x_of_t_end = W_of_t_convolve(t_reactivity, u_rect_reactivity, model_memory_alpha, model_memory_beta)
    y_of_t = u_merge / (1 + x_of_t ** 2)
    fpath = NB_OUTPUT + os.sep + 'uy_stack_comb_identity'
    
    _ = plot_uy_stack(t_merge, u_merge, y_of_t, '', fpath, simplify=False, figsize=(5, 2))
    
    fpath = NB_OUTPUT + os.sep + 'uxrp_model'
    title = r'Wiener model dynamics ($\alpha=%.1f,\beta=%.1f, p_0=%.2f$, vary $T$)' % (model_memory_alpha, model_memory_beta, model_jump_p0)
    #_ = plot_uy_stack(t, u, y_of_t, '', fpath, simplify=False, figsize=(5,2))
    
    r_of_t = 1 / (1 + x_of_t ** 2)
    p_of_t = model_jump_p0 * r_of_t * u_merge / 100
    plot_uxrp_stack(t_merge, u_merge/100, x_of_t, r_of_t, p_of_t, title, fpath) 
    return x_of_t, y_of_t, r_of_t, p_of_t

In [ ]:
x_of_t, y_of_t, r_of_t, p_of_t = ode_response_given_t_u_theta_v1(t_merge, u_merge, model_memory_alpha, model_memory_beta, model_jump_p0, nmult=5)

In [ ]:
t_at_pulses_hab = np.array(tmid_list) - 0.99 * duty_hab * period_S1_hab

t_at_pulses_reactivity = np.arange(
    period_S1_reactivity, 
    period_S1_reactivity * num_pulse_reactivity + period_S1_reactivity, 
    period_S1_reactivity
) - 0.99 * duty_reactivity * period_S1_reactivity + t[-1]

#t_at_pulses_reactivity = np.arange(t[-1] + period_S1_reactivity, 
#                                   t[-1] + period_S1_reactivity * num_pulse_reactivity + period_S1_reactivity, 
#                                   period_S1_reactivity)
t_at_pulses_merge = np.concatenate(
    [t_at_pulses_hab,
    t_at_pulses_reactivity]
)

indices = np.searchsorted(t_merge, t_at_pulses_merge)
t_sub = t_merge[indices]
x_sub = x_of_t[indices]
print(len(t_sub))

nrepeats = num_flies  # ensemble size

assert len(t_sub) in [350, 1050]  # 3*100 hab expt + 50  --or-- 5*200 hab expt + 50

pflips_variable_numeric = model_jump_p0 * 1 / (1 + x_sub ** 2)

plt.plot(t_sub, pflips_variable_numeric, '--ok')
plt.title('p_jump(t) numeric (from ODE)')

## Now compare above plot for discretized "numeric" and "heuristic" pflips_variable options

In [ ]:
# heuristic for x_low, x_high of x(t) trajectory - assumes periodic sequence of u(t) input pulses
def get_traj_xhigh_xlow(alpha, beta, period_S1, duty, amplitude, nperiods):
    pulse_area = amplitude * period_S1 * duty
    
    q = np.exp(- alpha * period_S1)
    Q = beta * pulse_area * (1 - q ** duty) / (alpha * period_S1 * duty)
    
    arr_n = np.arange(0, nperiods)
    
    arr_xhigh = Q * (1- q ** (arr_n+1)) / (1 - q)
    #arr_xhigh = Q * (1- q ** arr_n) / (1 - q)
    arr_xlow = np.zeros_like(arr_xhigh)
    arr_xlow[1:] = arr_xhigh[:-1] * q ** (1 - duty)
    
    return arr_n, arr_xhigh, arr_xlow


# heuristic for x(t) -- assumes u(t) is Dirac comb (delta-fn pulses) of area 1.0
def get_traj_x_deltafn(alpha, beta, period_S1, nperiods):
    pulse_area = 1.0
    
    q = np.exp(- alpha * period_S1)
    Q = beta * pulse_area 
    
    arr_n = np.arange(0, nperiods)
    arr_x_for_delta = Q * (1- q ** (arr_n+1)) / (1 - q)
    
    return arr_n, arr_x_for_delta


def sigma_of_x(x, N=2):
    return 1 / (1 + x ** N)

#### Now construct the likelihood function (timeseries) for bayesian inference

In [ ]:
# pick alpha, beta, p0 
heuristic_alpha   = model_memory_alpha
heuristic_beta    = model_memory_beta
heuristic_jump_p0 = model_jump_p0

# stimulus settings for heuristic curves pflips_variable
# ==================================================================
period_S1_A = 1.0
period_S1_C = 5.0

duty_A =  0.01
duty_C =  0.01 / period_S1_C
amplitude = 100.0

nperiods_A = 200
nperiods_C = 50

# construct x(t), p_jump(t) (heuristic for discretized values at each stimulus)
# ==================================================================
arr_n_1, arr_xhigh_1, arr_xlow_1 = get_traj_xhigh_xlow(heuristic_alpha, heuristic_beta, period_S1_A, duty_A, amplitude, nperiods_A)
arr_n_3, arr_xhigh_3, arr_xlow_3 = get_traj_xhigh_xlow(heuristic_alpha, heuristic_beta, period_S1_C, duty_C, amplitude, nperiods_C)

_, arr_x_delta_1 = get_traj_x_deltafn(heuristic_alpha, heuristic_beta, period_S1_A, nperiods_A)
_, arr_x_delta_3 = get_traj_x_deltafn(heuristic_alpha, heuristic_beta, period_S1_C, nperiods_C)

arr_y1 = heuristic_jump_p0 * sigma_of_x(arr_xlow_1)  
arr_y3 = heuristic_jump_p0 * sigma_of_x(arr_xlow_3) 

arr_y1_alt = heuristic_jump_p0 * sigma_of_x(arr_xhigh_1)  
arr_y3_alt = heuristic_jump_p0 * sigma_of_x(arr_xhigh_3)

arr_y1_delta = heuristic_jump_p0 * sigma_of_x(arr_x_delta_1)
arr_y3_delta = heuristic_jump_p0 * sigma_of_x(arr_x_delta_3)

# this defines the likelihood function 
pflips_variable_heuristic = np.concatenate([arr_y1_delta for a in range(5)] + [arr_y3_delta])

In [ ]:
ls_kwargs_loc = dict(
    markersize=6,
    marker='o',
    linestyle='-',
    linewidth=1,
)

label1 = r'$T=1$'
label3 = r'$T=5$'

z1, z3 = 10, 11

# plot A
# ==================================================================
plt.close('all')
plt.plot(arr_n_1, arr_xhigh_1, c='red', label='T=1, x-high', zorder=z1, **ls_kwargs_loc)
plt.plot(arr_n_1, arr_xlow_1, c='green', label='T=1, x-low', zorder=z1, **ls_kwargs_loc)
plt.plot(arr_n_1, arr_x_delta_1, c='blue', label=r'T=1, $\delta$ input', zorder=z1+5, marker='o', markersize=2)

plt.plot(arr_n_3, arr_xhigh_3, c='red', label='T=5, x-high', zorder=z3, **ls_kwargs_loc)
plt.plot(arr_n_3, arr_xlow_3, c='green', label='T=5, x-low', zorder=z3, **ls_kwargs_loc)
plt.plot(arr_n_3, arr_x_delta_3, c='blue', label=r'T=5 $\delta$ input', zorder=z1+5, marker='o', markersize=2)

plt.axvline(nperiods_A, linestyle='--', color='k')
plt.axvline(nperiods_C, linestyle='--', color='k')
plt.title('Heuristic x[k]-values for different stimulus periods')
plt.legend()
plt.show()

# plot B
# ==================================================================
plt.close('all')
plt.plot(arr_n_1, arr_y1, c='green', label=label1, zorder=z1, **ls_kwargs_loc)
plt.plot(arr_n_3, arr_y3, c='green', label=label3, zorder=z3, **ls_kwargs_loc)

plt.plot(arr_n_1, arr_y1_alt, c='red', label=label1 + ' (x-high)', alpha=0.5, zorder=z1, **ls_kwargs_loc)
plt.plot(arr_n_3, arr_y3_alt, c='red', label=label3 + ' (x-high)', alpha=0.5, zorder=z3, **ls_kwargs_loc)

plt.plot(arr_n_1, arr_y1_delta, c='blue', label=label1 + ' (delta)', zorder=z1+5, marker='o', markersize=2)
plt.plot(arr_n_3, arr_y3_delta, c='blue', label=label3 + ' (delta)', zorder=z3+5, marker='o', markersize=2)

plt.axvline(nperiods_A, linestyle='--', color='k')
plt.axvline(nperiods_C, linestyle='--', color='k')

plt.plot(pflips_variable_numeric[0:200], '--ok', label='T=1, numeric (from ODE)', markersize=3, zorder=20)
plt.plot(pflips_variable_numeric[-50:], '--ok', label='T=5, numeric (from ODE)', markersize=3, zorder=20)
         
#plt.suptitle(r'Expect higher $f$ (lower $T$) implies faster TTH_discrete')
plt.title(r'Heuristic $p_{jump}[k]$ for different $x[k]$ choices (alpha=%s, beta=%s)' % (heuristic_alpha, heuristic_beta), fontsize=10)
plt.xlabel(r'period index $k$')
plt.ylabel(r'peak values $y[k]$')
plt.legend()
plt.grid(alpha=0.4)
#plt.tight_layout()

#plt.gca().set_xticks([0, 10, 20, 30, 40, 50, 60])
#plt.gca().set_xticklabels([-30, -20, -10, 0, 10, 20, 30])
plt.xlabel(r'num. pulses applied')
plt.show()

## Plot synthetic data which was generated using dynamic $p_{jump}(t)$ 

In [ ]:
nrepeats = 128

#### A - Here "pflips_variable" is the numeric version found from solving an ODE and sampling the output at regular time intervals

In [ ]:
jump_arr = gen_jumps_from_prob_flips(len(t_sub), nrepeats, pflips_variable_numeric)

plt.close()
fig = plt.figure(figsize=(8,8))
gs_main = GridSpec(2, 1, height_ratios=[1, 0.5], hspace=0.2)

gs1 = GridSpecFromSubplotSpec(1, 1, subplot_spec=gs_main[0])
gs2 = GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[1], hspace=.15)

ax0 = fig.add_subplot(gs1[0])
ax1 = fig.add_subplot(gs2[0], sharex=ax0)
ax2 = fig.add_subplot(gs2[1], sharex=ax0)

ax0.imshow(jump_arr, interpolation='none', aspect='auto')
ax0.set_title(r'Synthetic data (5x200 @ $T=1s$, 50 @ $T=5s$)')

#ax1.plot(t_sub, pflips_variable, '-ok', markersize=3, label=r'hidden $p_{jump}$')
ax1.plot(pflips_variable_numeric, '-og', markersize=3, label=r'hidden $p_{jump}$')
ax1.legend()
ax1.set_ylim(-0.01,1.05)
#ax1.set_xlim(-0.5, 50 - 0.5)
ax1.set_title(r'$p_{jump}$ habituation (high $%.2f$, low $%.2f$, end $%.2f$)' % (pflips_variable_numeric[0], np.min(pflips_variable_numeric), pflips_variable_numeric[-1]))
#ax1.set_xticks([])
ax1.tick_params(labelbottom=False)

#ax2.plot(t_sub, np.mean(jump_arr, axis=0), '--o', markersize=3, color='#4D4DFF', label=r'mean $\langle z_k \rangle$')
#ax2.plot(np.mean(jump_arr, axis=0), '--o', markersize=3, color='#4D4DFF', label=r'mean $\langle z_k \rangle$')
ax2.plot(np.mean(jump_arr, axis=0), '--o', markersize=3, color='k', label=r'mean $\langle z_k \rangle$')
ax2.legend()
ax2.set_ylim(-0.01,1.05)
#ax2.set_xlim(-0.5, 50 - 0.5)

ax2.set_xlabel(r'pulse index $k$')

#gs.tight_layout(fig, rect=[0, 0, 1, 0.97])
plt.show()
print('jump_arr.shape', jump_arr.shape)
print('saving jumps_arr to...', NB_OUTPUT + os.sep + 'jumpdata_ode.csv')
np.savetxt(NB_OUTPUT + os.sep + 'jumpdata_ode.csv', jump_arr, fmt='%d')

#### B - Here "pflips_variable" is built from analytic formula that assumes the input is a sum of delta functions

In [ ]:
pflips_variable_heuristic = np.concatenate([arr_y1_delta for a in range(5)] + [arr_y3_delta])

jump_arr = gen_jumps_from_prob_flips(len(t_sub), nrepeats, pflips_variable_heuristic)

plt.close()
fig = plt.figure(figsize=(8,8))
gs_main = GridSpec(2, 1, height_ratios=[1, 0.5], hspace=0.2)

gs1 = GridSpecFromSubplotSpec(1, 1, subplot_spec=gs_main[0])
gs2 = GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[1], hspace=.15)

ax0 = fig.add_subplot(gs1[0])
ax1 = fig.add_subplot(gs2[0], sharex=ax0)
ax2 = fig.add_subplot(gs2[1], sharex=ax0)

ax0.imshow(jump_arr, interpolation='none', aspect='auto')
ax0.set_title(r'Synthetic data (5x200 @ $T=1s$, 50 @ $T=5s$) - use analytic $p_{jump}$')

#ax1.plot(t_sub, pflips_variable, '-ok', markersize=3, label=r'hidden $p_{jump}$')
ax1.plot(pflips_variable_heuristic, '-og', markersize=3, label=r'hidden $p_{jump}$')
ax1.legend()
ax1.set_ylim(-0.01,1.05)
#ax1.set_xlim(-0.5, 50 - 0.5)
ax1.set_title(r'$p_{jump}$ habituation (high $%.2f$, low $%.2f$, end $%.2f$)' % (pflips_variable_heuristic[0], np.min(pflips_variable_heuristic), pflips_variable_heuristic[-1]))
#ax1.set_xticks([])
ax1.tick_params(labelbottom=False)

#ax2.plot(t_sub, np.mean(jump_arr, axis=0), '--o', markersize=3, color='#4D4DFF', label=r'mean $\langle z_k \rangle$')
#ax2.plot(np.mean(jump_arr, axis=0), '--o', markersize=3, color='#4D4DFF', label=r'mean $\langle z_k \rangle$')
ax2.plot(np.mean(jump_arr, axis=0), '--o', markersize=3, color='k', label=r'mean $\langle z_k \rangle$')
ax2.legend()
ax2.set_ylim(-0.01,1.05)
#ax2.set_xlim(-0.5, 50 - 0.5)

ax2.set_xlabel(r'pulse index $k$')

#gs.tight_layout(fig, rect=[0, 0, 1, 0.97])
plt.show()
print('jump_arr.shape', jump_arr.shape)
print('saving jumps_arr to...', NB_OUTPUT + os.sep + 'jumpdata_ode_deltafn.csv')
np.savetxt(NB_OUTPUT + os.sep + 'jumpdata_ode_deltafn.csv', jump_arr, fmt='%d')

#### Now compare ODE sim. vs expt data (means)

In [ ]:
synthetic_means = np.mean(jump_arr, axis=0)

plt.figure(figsize=(7,2))
plt.plot(np.arange(n), jumps_sums, 
         '--ok', linewidth=1, markersize=2, label='data (GD)')
plt.plot(np.arange(n), synthetic_means, 
         '--o', color='grey', linewidth=1, markersize=2, label='model')
#plt.xticks(fontsize=8)
#plt.yticks(fontsize=8)
plt.legend()
plt.ylim(0,1)
plt.xlabel(r'pulse index $k$')

## Alternate link functions mapping $x(t)$ to a probability of response (bernoulli parameter $p$)
- originally had form $p(t) = p_0 \sigma(x)$ with $\sigma(x)=1/(1+x^2)$
- in general, we want $f(x)$ such that
    1. $f(x)$ is bounded between 0 and 1
    2. $f(0) = 1$
    3. $f(x)$ is non-increasing with $x$ on $[0, \infty)$   

In [ ]:
def foo_A(x, N=2):
    return 1 / (1 + x ** N)

def foo_exp(x, r=1.0):
    return np.exp(-r*x)

def foo_B(x, r=1.0):
    return 2 * np.exp(-r*x) / (np.exp(r*x) + np.exp(-r*x))

def foo_C(x, r=1.0):
    return 2 * np.exp(-r*x) / (1 + np.exp(-r*x))

x_check = np.linspace(0, 10, 200)

plt.figure(figsize=(2,2))
for N in [1,2,3,4]:
    plt.plot(x_check, foo_exp(x_check, N))
plt.title(r'$f(x)=e^{-rx}$')
plt.xlabel(r'$x$');plt.ylabel(r'$p_{jump}$');plt.show()

plt.figure(figsize=(2,2))
for N in [1,2,3,4]:
    plt.plot(x_check, foo_A(x_check, N))
plt.title(r'$f(x)=1 / (1 + x^N)$')
plt.xlabel(r'$x$');plt.ylabel(r'$p_{jump}$');plt.show()

plt.figure(figsize=(2,2))
for r in [0.1, 0.5, 1,2,3,4]:
    plt.plot(x_check, foo_B(x_check, r))
plt.title(r'$f(x)=2 \frac{e^{-rx}}{e^{rx} + e^{-rx}}$')
plt.xlabel(r'$x$');plt.ylabel(r'$p_{jump}$');plt.show()

plt.figure(figsize=(2,2))
for r in [0.1, 0.5, 1,2,3,4]:
    plt.plot(x_check, foo_C(x_check, r))
plt.title(r'$f(x)=2 \frac{e^{-rx}}{1 + e^{-rx}}$')
plt.xlabel(r'$x$');plt.ylabel(r'$p_{jump}$');plt.show()